In [10]:
import pandas as pd

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option("display.max_colwidth", None)

In [2]:
import pandas as pd

file_path = "/home/user/Downloads/app_invest_dubai_ded_2026_09_01.xlsx"

df = pd.read_excel(file_path)

print(df.head())

                                               url listing_dul_number  \
0  https://app.invest.dubai.ae/dul/dul-EB3367?bk=1             EB3367   
1  https://app.invest.dubai.ae/dul/dul-DJ4452?bk=1             DJ4452   
2  https://app.invest.dubai.ae/dul/dul-FA7203?bk=1             FA7203   
3  https://app.invest.dubai.ae/dul/dul-AS6704?bk=1             AS6704   
4  https://app.invest.dubai.ae/dul/dul-BS4196?bk=1             BS4196   

  listing_license_number                        listing_issuing_authority  \
0                  55734  Dubai Integrated Economic Zones Authority - DSO   
1                  43179  Dubai Integrated Economic Zones Authority - DSO   
2             2647755.01                    Meydan City Corporation - MCC   
3                 629393          Department of Economy and Tourism - DET   
4                1017286          Department of Economy and Tourism - DET   

                       listing_business_name_english  \
0               HERITORS PROPERTY CONSULTA

In [3]:
df.shape

(134295, 17)

In [4]:
df.isnull().sum()

url                                  0
listing_dul_number                   0
listing_license_number               0
listing_issuing_authority            0
listing_business_name_english       13
listing_business_name_arabic         1
dul_number                           0
license_status                   67360
expiry_date                      67360
license_number                   67360
business_name                    67360
license_type                     67364
issuing_authority                67360
legal_type                       67842
issue_date                       67360
address                          67526
license_activities               67534
dtype: int64

In [21]:
df[df.listing_business_name_arabic.isna()].url

105046    https://app.invest.dubai.ae/dul/dul-DP8065?bk=1
Name: url, dtype: object

In [7]:
df.nunique()

url                              134295
listing_dul_number               134295
listing_license_number           134204
listing_issuing_authority            14
listing_business_name_english    132635
listing_business_name_arabic     132197
dul_number                       134295
license_status                        7
expiry_date                        2171
license_number                    66921
business_name                     66222
license_type                          4
issuing_authority                    13
legal_type                           20
issue_date                         6549
address                           54619
license_activities                15601
dtype: int64

In [8]:
df.value_counts()

url                                              listing_dul_number  listing_license_number  listing_issuing_authority                listing_business_name_english                 listing_business_name_arabic          dul_number  license_status            expiry_date  license_number  business_name                                 license_type  issuing_authority                        legal_type                                          issue_date  address                                                                                                             license_activities                                                                                                                               
https://app.invest.dubai.ae/dul/dul-FI1787?bk=1  FI1787              1648098                 Department of Economy and Tourism - DET  G X B PROPERTIES L.L.C                        جي اكس بي للعقارات ذ.م.م              FI1787      Active                    2027-08-27   1648098         G X

In [12]:
df.license_status.value_counts()

license_status
Active                      49695
under admin cancellation    13764
Cancelled                    2424
Frozen                        547
Liquidated                    408
Under Transaction              96
Under Construction              1
Name: count, dtype: int64

In [14]:
import pandas as pd

text_cols = df.select_dtypes(include="object").columns

issues = {}

for col in text_cols:
    mask = (
        df[col]
        .fillna("")
        .astype(str)
        .str.contains(r"(?<!\\)\|", regex=True)
    )

    if mask.any():
        temp = df.loc[mask, ["url", col]].copy()
        temp.insert(0, "row_no", temp.index)  # DataFrame row number
        issues[col] = temp

# Print results
if issues:
    print("Columns containing unescaped '|':\n")

    for col, data in issues.items():
        print(f"\n=== {col} ({len(data)} rows) ===")
        print(data.to_string(index=False))
else:
    print("No unescaped '|' found.")

Columns containing unescaped '|':


=== address (6 rows) ===
 row_no                                             url                                                                                              address
    569 https://app.invest.dubai.ae/dul/dul-DR4166?bk=1                        مكتب رقم 9|720 - ملك مدينه دبى الصناعيه - سيح شعيب 2, Office 720, Saih Shuaib
   4456 https://app.invest.dubai.ae/dul/dul-AP2093?bk=1     مكتب رقم 801 ملك الانصاري العقارية |ملك خاص | - البرشاء الأولى - بردبي, Office 801, Al Barshaa 1
  14451 https://app.invest.dubai.ae/dul/dul-BH1054?bk=1                محل رقم |G08 ملك مشاريع شمال دبي لاند ش ذ م م - وادي الصفا 7, Shop 08, Wadi Al Safa 7
  20752 https://app.invest.dubai.ae/dul/dul-BF5498?bk=1                         |محل S4ملك -بلدية دبي -جبل علي الصناعيه الأولى, Shop 4, Jebel Ali Industrial
  28226 https://app.invest.dubai.ae/dul/dul-CK2332?bk=1                  |مكتب رقم2402c-19 ملك سالم احمد سالم القيسى-الخليج التجاري, Office 19, Burj Khali

In [15]:
import pandas as pd

escape_pattern = r'[\n\r\t\b\f\v]'

issues = []


for col in df.select_dtypes(include="object").columns:
    mask = df[col].fillna("").astype(str).str.contains(
        escape_pattern,
        regex=True,
        na=False
    )

    if mask.any():
        temp = df.loc[mask, [col]].copy()
        temp.insert(0, "row_no", temp.index + 2)
        temp["column"] = col
        temp["issue_value"] = temp[col]

        if "pdp_url" in df.columns:
            temp["pdp_url"] = df.loc[mask, "pdp_url"]

        issues.append(
            temp[["row_no", "pdp_url", "column", "issue_value"]]
        )

if issues:
    result = pd.concat(issues, ignore_index=True)

    print("=== Escape Character Issues ===")
    print(result.to_string(index=False))
    print(f"\nTotal issues: {len(result)}")
else:
    print("No escape characters found.")

No escape characters found.


In [17]:
import re

url_pattern = re.compile(
    r"^https?://[^\s/$.?#].[^\s]*$",
    re.IGNORECASE
)

invalid_urls = df[
    
    df["url"].isna() |
    ~df["url"].astype(str).str.match(url_pattern)
]

print(f"Invalid URLs: {len(invalid_urls)}")
print(invalid_urls[[ "url"]])

Invalid URLs: 0
Empty DataFrame
Columns: [url]
Index: []


In [18]:
df.columns

Index(['url', 'listing_dul_number', 'listing_license_number',
       'listing_issuing_authority', 'listing_business_name_english',
       'listing_business_name_arabic', 'dul_number', 'license_status',
       'expiry_date', 'license_number', 'business_name', 'license_type',
       'issuing_authority', 'legal_type', 'issue_date', 'address',
       'license_activities'],
      dtype='object')

In [19]:
df.duplicated().sum()

np.int64(0)